# 00 - Setup do ambiente no Google Colab

Este notebook prepara o runtime do Colab para executar os experimentos do projeto:
1. Verifica a GPU (Tesla T4 ou superior),
2. Clona o repositório,
3. Instala as dependências (`requirements-colab.txt`),
4. Carrega o dataset ToLD-BR via Hugging Face (sem Google Drive),
5. Fornece uma célula para baixar resultados ao final da sessão,

## 1. Verificar GPU

In [28]:
!nvidia-smi

Sun Oct 18 17:35:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [29]:
import torch

print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU: ", torch.cuda.get_device_name(0))
    print("Versão do PyTorch: ", torch.__version__)
    print("CUDA: ", torch.version.cuda)

CUDA disponível: True
GPU:  Tesla T4
Versão do PyTorch:  2.11.0+cu130
CUDA:  13.0


## 2. Clonar o Repósitório

In [30]:
import os

REPO_URL = "https://github.com/elianisaac/Classifica-o-de-Discurso-T-xico-em-Redes-Sociais.git"
REPO_DIR = "Classifica-o-de-Discurso-T-xico-em-Redes-Sociais"

# Só clona se a pasta NÃO existir
if not os.path.exists(f"/content/{REPO_DIR}"):
    !git clone {REPO_URL} /content/{REPO_DIR}

# Força a entrada no caminho absoluto correto
%cd /content/{REPO_DIR}

/content/Classifica-o-de-Discurso-T-xico-em-Redes-Sociais


## 3. Instalar Dependencias

Usa `requirements-colab.txt`


In [31]:
# Atualiza as ferramentas de build do Python
!pip install --upgrade pip setuptools wheel

# Instala as dependencias
%pip install -q -r requirements-collab.txt

  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
Using cached setuptools-84.0.0-py3-none-any.whl (818 kB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu130 requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [32]:
# Verificação rápida das versões instaladas
import datasets, transformers, sklearn
print("datasets:", datasets.__version__)
print("transformers: ", transformers.__version__)
print("scikit-learn: ", sklearn.__version__)

datasets: 2.21.0
transformers:  5.18.0
scikit-learn:  1.6.1


# 4. Carregamento do Dataset

O script `src/load_data.py` baixa o dataset do Hugging Face (`JAugusto97/told-br`) e salva em disco (`data/processed/`) para evitar re-executar o *loading script* a cada uso.

In [33]:
!python src/load_data.py --variant binary --output data/processed/

[2026-10-18 17:36:04] [INFO] [load_data] ============================================================
[2026-10-18 17:36:04] [INFO] [load_data] Processando variante: binary
[2026-10-18 17:36:04] [INFO] [load_data] ============================================================
[2026-10-18 17:36:04] [INFO] [load_data] Carregando do disco: data/processed/told-br-binary
[2026-10-18 17:36:04] [INFO] [load_data] Resumo do dataset (binary):
[2026-10-18 17:36:04] [INFO] [load_data]   train: 16800 exemplos | features=['text', 'label']
[2026-10-18 17:36:04] [INFO] [load_data]     label_counts_label: {'0': 9425, '1': 7375}
[2026-10-18 17:36:04] [INFO] [load_data]   test: 2100 exemplos | features=['text', 'label']
[2026-10-18 17:36:04] [INFO] [load_data]     label_counts_label: {'1': 972, '0': 1128}
[2026-10-18 17:36:04] [INFO] [load_data]   validation: 2100 exemplos | features=['text', 'label']
[2026-10-18 17:36:04] [INFO] [load_data]     label_counts_label: {'1': 908, '0': 1192}
[2026-10-18 17:36:0

In [34]:
# Inspeção rápida do dataset carregado
from datasets import load_from_disk
ds = load_from_disk("data/processed/told-br-binary")
print(ds)
print("Exemplo: ", ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16800
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2100
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2100
    })
})
Exemplo:  {'text': 'rt @user olha quem chegouuuuu, nossos queridinhos!!! vem na minha direção 🤩\n\nfem 👧: 10/15\nmasc 👦: 25/40\n\n#qqmt https://t.co/jivoqeuq…', 'label': 0}


In [35]:
# Roda o pré-processamento gerando os textos limpos
!python src/preprocessing.py \
    --input data/processed/told-br-binary \
    --output data/processed/told-br-binary-clean

[2026-10-18 17:36:17] [INFO] [preprocessing] Carregando dataset de: data/processed/told-br-binary
[2026-10-18 17:36:17] [INFO] [preprocessing] Aplicando pré-processamento nas partições...
[2026-10-18 17:36:17] [INFO] [preprocessing]   Processando split: train
Map (num_proc=2): 100% 16800/16800 [00:01<00:00, 12103.27 examples/s]
[2026-10-18 17:36:18] [INFO] [preprocessing]   Processando split: test
Map (num_proc=2): 100% 2100/2100 [00:00<00:00, 7777.13 examples/s]
[2026-10-18 17:36:18] [INFO] [preprocessing]   Processando split: validation
Map (num_proc=2): 100% 2100/2100 [00:00<00:00, 8124.63 examples/s]
[2026-10-18 17:36:19] [INFO] [preprocessing] Salvando dataset processado em: data/processed/told-br-binary-clean
Saving the dataset (1/1 shards): 100% 16800/16800 [00:00<00:00, 1175463.04 examples/s]
Saving the dataset (1/1 shards): 100% 2100/2100 [00:00<00:00, 474775.68 examples/s]
Saving the dataset (1/1 shards): 100% 2100/2100 [00:00<00:00, 443685.19 examples/s]
[2026-10-18 17:36:19

## 5. Executar os experimentos

Com o ambiente pronto e o dataset carregado, rode os scripts de treino. Exemplos:"

In [ ]:
# Baseline: Regressão Logística
!python src/train_logistic.py --data data/processed/ --output results

python3: can't open file '/content/Classifica-o-de-Discurso-T-xico-em-Redes-Sociais/Classifica-o-de-Discurso-T-xico-em-Redes-Sociais/Classifica-o-de-Discurso-T-xico-em-Redes-Sociais/Classifica-o-de-Discurso-T-xico-em-Redes-Sociais/src/train_logistic.py': [Errno 2] No such file or directory


In [48]:
# BERTimbau (fine-tuning)
!python src/train_bertimbau.py
--data data/processed/
--model neuralmind/bert-base-portuguese-cased
--epochs 5
--batch-size 16
--output-dir results/models/

SyntaxError: invalid syntax (1041307888.py, line 3)

In [ ]:
# Avaliação e comparação
!python src/evaluate.py --results results/ --output results/metrics/

## 6. Baixar resultados ao final da sessão\n",

In [ ]:
import shutil
from google.colab import files

# Compacta a pasta de resultados
archive_path = shutil.make_archive("/content/results_backup", "zip", "/content/[nome-do-repo]/results")
print("Arquivo criado:", archive_path)

# Baixa o arquivo para o computador local
files.download(archive_path)